<a href="https://colab.research.google.com/github/antonellagambarte/deep-learning--CEIA/blob/main/Copia_de_GAMBARTE_ANTONELLA_NEREA_DL_TP2_Co21.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, time
from collections import Counter
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import transforms, datasets
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import itertools
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)
DATASET_ROOT_TRAIN = 'drive/MyDrive/dataset_emociones/train'
DATASET_ROOT_VAL   = 'drive/MyDrive/dataset_emociones/validation'
INFER_DIR          = 'drive/MyDrive/new_images'
OUTPUT_MODEL       = 'drive/MyDrive/best_emotion_model.pth'

Device: cpu


In [ ]:
IMG_SIZE = 64
CHANNELS = 3

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*CHANNELS, [0.5]*CHANNELS),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*CHANNELS, [0.5]*CHANNELS),
])

In [ ]:

train_dataset = datasets.ImageFolder(DATASET_ROOT_TRAIN, transform=train_transforms)
val_dataset   = datasets.ImageFolder(DATASET_ROOT_VAL, transform=val_transforms)

class_names = train_dataset.classes
num_classes = len(class_names)
print("Clases detectadas:", class_names)
print("Num clases:", num_classes)
print("Train size:", len(train_dataset), "Val size:", len(val_dataset))

# conteo por clase en train
labels_train = [s[1] for s in train_dataset.samples]
counts = Counter(labels_train)
counts_named = {class_names[i]: counts[i] for i in range(num_classes)}
print("Conteos (train):", counts_named)

Clases detectadas: ['alegria', 'disgusto', 'enojo', 'miedo', 'seriedad', 'sorpresa', 'tristeza']
Num clases: 7
Train size: 12271 Val size: 3068
Conteos (train): {'alegria': 4772, 'disgusto': 717, 'enojo': 705, 'miedo': 281, 'seriedad': 2524, 'sorpresa': 1290, 'tristeza': 1982}


In [ ]:
# LIMIT_PER_CLASS = 200

# import numpy as np
# from torch.utils.data import Subset

# def subset_per_class(dataset, limit):
#     targets = np.array([s[1] for s in dataset.samples])
#     indices = []
#     for cls in np.unique(targets):
#         cls_indices = np.where(targets == cls)[0]
#         # si hay menos imágenes que 'limit', toma todas las disponibles
#         cls_indices = cls_indices[:min(limit, len(cls_indices))]
#         indices.extend(cls_indices)
#     return Subset(dataset, indices)

# train_dataset = subset_per_class(train_dataset, LIMIT_PER_CLASS)
# val_dataset   = subset_per_class(val_dataset, LIMIT_PER_CLASS)



In [ ]:
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# #######

# print(train_dataset.classes)
# imgs, labels = next(iter(train_loader))
# print("Batch shape:", imgs.shape)
# print("Labels únicos:", labels.unique())

In [ ]:
# from torchvision.datasets import ImageFolder
# from torchvision import transforms

# train_dataset = ImageFolder(DATASET_ROOT_TRAIN, transform=transforms.ToTensor())
# print("Clases detectadas:", train_dataset.classes)
# #######

In [ ]:
def conv_block(c_in, c_out, k=3, p='same', s=1, pk=2):
    if p == 'same':
        padding = k//2
    else:
        padding = 0
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=k, stride=s, padding=padding),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=pk)
    )

class CNN(nn.Module):
    def __init__(self, n_channels=3, n_outputs=7):
        super().__init__()
        self.conv1 = conv_block(c_in = n_channels, c_out = 32, k=3, p='same', s=1, pk=2)
        self.conv1_out = None
        self.drop = nn.Dropout2d(p=0.4)
        self.conv2 = conv_block(c_in = 32, c_out = 64, k=3, p='same', s=1, pk=2)
        self.conv2_out = None
        self.conv3 = conv_block(c_in = 64, c_out = 128, k=3, p='same', s=1, pk=2)
        self.conv3_out = None

        # adaptive pool para fijar dimensiones a 3x3 independientemente del input
        self.adapt = nn.AdaptiveAvgPool2d((3,3))
        self.fc = nn.Linear(128*3*3, n_outputs)


        print('--- CNN creada ---')
        pytorch_total_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print('Parámetros entrenables:', pytorch_total_params)

    def forward(self, x):
        self.conv1_out = self.drop(self.conv1(x))
        self.conv2_out = self.drop(self.conv2(self.conv1_out))
        self.conv3_out = self.conv3(self.conv2_out)
        y = self.adapt(self.conv3_out)
        y = y.flatten(start_dim=1)
        y = self.fc(y)
        return y

model = CNN(n_channels=CHANNELS, n_outputs=num_classes).to(device)

--- CNN creada ---
Parámetros entrenables: 101767


In [ ]:
label_counts = np.array([counts[i] for i in range(num_classes)])
print("label_counts:", label_counts)

class_weights = 1.0 / (label_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * num_classes
print("class_weights (normalizadas):", np.round(class_weights,3))

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

label_counts: [4772  717  705  281 2524 1290 1982]
class_weights (normalizadas): [0.178 1.182 1.202 3.017 0.336 0.657 0.428]


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device); labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, acc, f1

def eval_model(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device); labels = labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, acc, f1, all_labels, all_preds

In [ ]:
# 8) Ejecutar entrenamiento
EPOCHS = 15
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'train_f1':[], 'val_f1':[]}
best_val_loss = np.inf

for ep in range(EPOCHS):
    t0 = time.time()
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, val_f1, _, _ = eval_model(model, val_loader, criterion)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc); history['val_acc'].append(val_acc)
    history['train_f1'].append(train_f1); history['val_f1'].append(val_f1)

    print(f"Ep {ep+1}/{EPOCHS}  train_loss {train_loss:.4f} acc {train_acc:.4f} f1 {train_f1:.4f}  |  val_loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f}  (t={(time.time()-t0):.1f}s)")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), OUTPUT_MODEL)
        print("  -> Guardado mejor modelo")


KeyboardInterrupt: 

In [ ]:
# 9) Plots: pérdida, accuracy, F1 por época (train vs val)
epochs = range(1, len(history['train_loss'])+1)
plt.figure(figsize=(14,4))
plt.subplot(1,3,1)
plt.plot(epochs, history['train_loss'], label='train'); plt.plot(epochs, history['val_loss'], label='val'); plt.title("Loss"); plt.legend()
plt.subplot(1,3,2)
plt.plot(epochs, history['train_acc'], label='train'); plt.plot(epochs, history['val_acc'], label='val'); plt.title("Accuracy"); plt.legend()
plt.subplot(1,3,3)
plt.plot(epochs, history['train_f1'], label='train'); plt.plot(epochs, history['val_f1'], label='val'); plt.title("F1 (macro)"); plt.legend()
plt.show()
